# Turkish Morph Retrieval — encoder baseline evaluation v2

Bu notebook varsayılan olarak **20-family GPT-5.6 Sol preview** üzerinde encoder'ları karşılaştırır.
Her family: `1 query + 1 positive + 8 hard negative + 2 easy negative`.

> **Önemli:** Bu 20 family yalnız deterministic QC'den geçti; bağımsız LLM judge ve insan
doğrulaması tamamlanmadı. Sonuçlar pipeline/model seçimi için pilot sinyaldir, paper sonucu değildir.

V2'nin katmanları: veri/artefakt kontrolü → ucuz baseline'lar → dense encoder'lar → hard-negative
ayrımı → bootstrap CI → paired testler → fenomen/slice analizi → ablation → hata analizi → export.
Full-corpus skorları yalnız insan pooling qrels'i bulunduğunda açılır.

## 0. Colab kullanımı

1. `Runtime > Change runtime type` ile GPU seçin (Qwen3-8B için A100 önerilir).
2. Aşağıdaki hücreleri sırayla çalıştırın.
3. Varsayılan üç model orta seviye GPU'da sırayla yüklenir ve belleği temizlenir.
4. Qwen3-8B varsayılan olarak kapalıdır; model tablosunda `enabled=True` yapabilirsiniz.
5. Çalışma yarıda kesilirse sonuç cache'i sayesinde tamamlanan modeller yeniden koşmaz.

In [ ]:
%pip -q install -U "sentence-transformers>=3.0,<6" "transformers>=4.48,<5" scikit-learn pandas matplotlib seaborn

In [ ]:
import gc, hashlib, json, os, random, shutil, subprocess, sys, time, traceback
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

REPO_URL = "https://" + "github.com/Bur8300/turkish-morph-retrieval.git"
candidates = [Path.cwd(), Path("/content/turkish-morph-retrieval")]
ROOT = next((p.resolve() for p in candidates if (p / "test" / "evaluation.py").exists()), None)
if ROOT is None:
    ROOT = Path("/content/turkish-morph-retrieval")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
sys.path.insert(0, str(ROOT))

from sentence_transformers import SentenceTransformer
from test.evaluation import (
    EVALUATION_API_VERSION, ablate_items, approximate_randomization, artifact_baseline_runs, build_pool_rows,
    bootstrap_ci, candidate_only_classifier, closed_qrels, evaluate_artifacts, evaluate_run,
    full_corpus_sparse_runs, holm_adjust, load_items, load_qrels, mcnemar, paired_bootstrap, score_encoder, slice_summary,
)
assert EVALUATION_API_VERSION == "2.1", "Repo eski. V2 dosyalarını push/pull edip runtime'ı yeniden başlatın."
print("repo:", ROOT)
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 1. Deney ayarları

`preview20`, şu an inceleyeceğiniz 20 örneği kullanır. Final veri freeze edildikten sonra
`frozen` moduna geçin. Sealed test üstünde model/prompt seçimi yapmayın; ayarlar development
üzerinde dondurulduktan sonra sealed test yalnız final rapor için bir kez çalıştırılmalı.

In [ ]:
DATASET_MODE = "preview20"          # "preview20" veya "frozen"
PREVIEW_FILE = ROOT / "test/previews/sol_preview_20_v31/preview_internal.json"
RUN_ID = "test_v31"
DEV_FILE = ROOT / f"test/runs/{RUN_ID}/release/morph_dev_v3.1.0.json"
TEST_FILE = ROOT / f"test/runs/{RUN_ID}/private/morph_test_internal_v3.1.0.json"
POOLED_QRELS = ROOT / f"test/runs/{RUN_ID}/private/pooled_human_qrels.tsv"

BATCH_SIZE = 16
N_BOOT = 10_000
USE_CACHE = True
OUTPUT_DIR = Path("/content/morph_eval_v2") if Path("/content").exists() else ROOT / "test/results/morph_eval_v2"
CACHE_DIR = OUTPUT_DIR / "cache"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
if DATASET_MODE == "preview20":
    DATA_FILE = PREVIEW_FILE
    DEV = []
    EVAL_ITEMS = load_items(DATA_FILE)
    assert len(EVAL_ITEMS) == 20, f"20 preview bekleniyordu, {len(EVAL_ITEMS)} bulundu"
    DATA_NOTICE = "PREVIEW: judge/insan doğrulaması yok; aynı 20 üstünde yalnız pilot karşılaştırma."
elif DATASET_MODE == "frozen":
    DATA_FILE = TEST_FILE
    DEV = load_items(DEV_FILE)
    EVAL_ITEMS = load_items(TEST_FILE)
    DATA_NOTICE = "FROZEN: 100 development / 500 sealed protokolünü koruyun."
else:
    raise ValueError("DATASET_MODE preview20 veya frozen olmalı")

DATA_SHA256 = hashlib.sha256(DATA_FILE.read_bytes()).hexdigest()
print(DATA_NOTICE)
print(f"family={len(EVAL_ITEMS)} | candidate={sum(len(x['candidates']) for x in EVAL_ITEMS)}")
print("data sha256:", DATA_SHA256)

## 2. Veri bütünlüğü ve dağılım

In [ ]:
integrity_rows = []
for item in EVAL_ITEMS:
    roles = Counter(c["role"] for c in item["candidates"])
    ids = [c["id"] for c in item["candidates"]]
    integrity_rows.append({
        "family_id": item["family_id"],
        "candidate_n": len(ids),
        "positive_n": roles["positive"],
        "hard_n": roles["hard_negative"],
        "easy_n": roles["easy_negative"],
        "unique_ids": len(ids) == len(set(ids)),
        "gold_exists": item["gold_id"] in ids,
    })
integrity = pd.DataFrame(integrity_rows)
display(integrity)
assert integrity[["unique_ids", "gold_exists"]].all().all()
assert (integrity[["candidate_n", "positive_n", "hard_n", "easy_n"]] == [11, 1, 8, 2]).all().all()
print("✓ Tüm family'ler 1/8/2 şemasına uyuyor.")

In [ ]:
fields = [
    "target_split", "query_sentence_count", "passage_sentence_count", "layer", "objective",
    "generalization_bucket", "macro_phenomenon", "target_feature",
]
for field in fields:
    counts = pd.Series([str(item.get(field)) for item in EVAL_ITEMS]).value_counts().rename("n").to_frame()
    counts["ratio"] = counts["n"] / len(EVAL_ITEMS)
    print("\n", field)
    display(counts)

## 3. Metrikler nasıl okunmalı?

**Ana morfoloji metrikleri:**

- `hard_only_recall@1`: gold, positive + 8 hard aday arasında birinci mi? Chance ≈ %11.1.
- `pairwise_hard_accuracy`: 160 gold–hard çiftinin ne kadarında gold daha yüksek skor aldı? Chance %50.
- `contrast_consistency`: gold, minimal morfolojik negatifi geçti mi?
- `hardest_hard_margin`: gold skoru − en yüksek hard skoru. Pozitif değer iyi.

**11 adaylık closed-family metrikleri:** `Recall@1/5/10`, `MRR@10`, `nDCG@10`, `MAP@10`.
Her query'de tek gold olduğu için `MAP@10 = MRR@10`; ikisini raporlamak hata değil ama aynı bilgidir.
`Recall@100` bu küçük havuzda ayırt edici değildir; esas kullanımı pooled full-corpus deneyidir.

## 4. Query-blind / ucuz artefakt baseline'ları

In [ ]:
qrels = closed_qrels(EVAL_ITEMS)
if DATASET_MODE == "frozen":
    artifact_summaries = evaluate_artifacts(DEV, EVAL_ITEMS)
    try:
        artifact_summaries["candidate_only_char_tfidf"] = candidate_only_classifier(DEV, EVAL_ITEMS)["summary"]
    except Exception as exc:
        print("candidate-only classifier atlandı:", exc)
else:
    artifact_summaries = {
        name: evaluate_run(qrels, run)[0]
        for name, run in artifact_baseline_runs(EVAL_ITEMS, learned_position=None).items()
    }
    print("Position-only ve candidate-only, ayrı development olmadan dürüstçe öğrenilemeyeceği için preview'da atlandı.")

ARTIFACT_DF = pd.DataFrame(artifact_summaries).T.sort_values("recall@1", ascending=False)
display(ARTIFACT_DF[["recall@1", "recall@5", "mrr@10", "ndcg@10", "map@10", "mean_rank"]].round(4))
print("closed R@1 chance =", round(1 / 11, 4))

In [ ]:
ax = ARTIFACT_DF["recall@1"].sort_values().plot.barh(figsize=(8, 4), color="#7a9cc6")
ax.axvline(1 / 11, color="black", linestyle="--", label="chance 1/11")
ax.set(title="Ucuz baseline Recall@1", xlabel="Recall@1", ylabel="")
ax.legend(); plt.tight_layout(); plt.show()

## 5. Encoder kayıt defteri

Varsayılanlar A100 icin mevcut proje listesindeki 5 encoder'dir. Bir model yüklenemezse deney durmaz; hata kaydedilir.
Prefix'ler model ailesinin retrieval biçimine göre ayrı tutulur. Qwen3-8B belleği yüksek olduğu
için A100/L4 disi runtime'larda kapatabilirsiniz. Aynı notebook'u tekrar çalıştırırken model veya prefix değişirse cache anahtarı da değişir.

In [ ]:
INSTRUCT = "Instruct: Given a Turkish web search query, retrieve relevant passages that answer the query\nQuery: "
MODEL_SPECS = [
    {"name": "e5-large", "repo": "intfloat/multilingual-e5-large", "query_prefix": "query: ", "document_prefix": "passage: ", "enabled": True},
    {"name": "bge-m3", "repo": "BAAI/bge-m3", "query_prefix": "", "document_prefix": "", "enabled": True},
    {"name": "modernbert-tr", "repo": "ytu-ce-cosmos/modernbert-tr-embed", "query_prefix": INSTRUCT, "document_prefix": "", "enabled": True},
    {"name": "trmteb-ft-110m", "repo": "trmteb/turkish-embedding-model-fine-tuned", "query_prefix": "", "document_prefix": "", "enabled": True},
    {"name": "qwen3-8b", "repo": "Qwen/Qwen3-Embedding-8B", "query_prefix": INSTRUCT, "document_prefix": "", "enabled": True, "dtype": "float16"},
]
display(pd.DataFrame(MODEL_SPECS)[["name", "repo", "enabled", "query_prefix", "document_prefix"]])

In [ ]:
EVAL_CODE_SHA256 = hashlib.sha256((ROOT / "test/evaluation.py").read_bytes()).hexdigest()

def cache_path(spec):
    payload = json.dumps({"spec": spec, "data": DATA_SHA256, "eval": EVAL_CODE_SHA256}, sort_keys=True)
    key = hashlib.sha256(payload.encode()).hexdigest()[:12]
    return CACHE_DIR / f"{spec['name']}-{key}.json"

def load_model(spec):
    kwargs = {"trust_remote_code": True}
    if spec.get("dtype") and torch.cuda.is_available():
        kwargs["model_kwargs"] = {"torch_dtype": getattr(torch, spec["dtype"])}
    model = SentenceTransformer(spec["repo"], **kwargs)
    try:
        model.default_prompt_name = None
    except Exception:
        pass
    return model

def run_one_model(spec):
    path = cache_path(spec)
    if USE_CACHE and path.exists():
        print(spec["name"], "cache'ten yüklendi")
        return json.loads(path.read_text()), {"model": spec["name"], "cached": True}
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    model = load_model(spec)
    load_seconds = time.perf_counter() - started
    dimension = model.get_sentence_embedding_dimension()
    score_started = time.perf_counter()
    result = score_encoder(
        model, EVAL_ITEMS, spec.get("query_prefix", ""), spec.get("document_prefix", ""),
        full_corpus=False, batch_size=BATCH_SIZE, include_full_run=True,
    )
    score_seconds = time.perf_counter() - score_started
    peak_gb = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
    path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    del model
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return result, {
        "model": spec["name"], "cached": False, "dimension": dimension,
        "load_seconds": load_seconds, "score_seconds": score_seconds, "peak_gpu_gb": peak_gb,
    }

## 6. Encoder'ları çalıştır

In [ ]:
RESULTS, RUNTIME_ROWS, MODEL_ERRORS = {}, [], []
for spec in [spec for spec in MODEL_SPECS if spec["enabled"]]:
    print("\n===", spec["name"], "===")
    try:
        RESULTS[spec["name"]], runtime = run_one_model(spec)
        RUNTIME_ROWS.append(runtime)
    except Exception as exc:
        MODEL_ERRORS.append({"model": spec["name"], "error": repr(exc), "traceback": traceback.format_exc()})
        print("ATLANDI:", repr(exc))
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
assert RESULTS, "Hiçbir model tamamlanmadı. MODEL_ERRORS çıktısını inceleyin."
display(pd.DataFrame(RUNTIME_ROWS))
if MODEL_ERRORS: display(pd.DataFrame(MODEL_ERRORS)[["model", "error"]])

## 7. Ana sonuç tablosu

In [ ]:
SUMMARY_DF = pd.DataFrame({name: result["summary"] for name, result in RESULTS.items()}).T
primary = [
    "hard_only_recall@1", "hard_only_mrr@10", "pairwise_hard_accuracy", "contrast_consistency",
    "hardest_hard_margin", "recall@1", "recall@5", "mrr@10", "ndcg@10", "map@10",
    "mean_rank", "mean_hard_rank",
]
display(SUMMARY_DF[primary].sort_values("hard_only_recall@1", ascending=False).round(4))
print("hard-only R@1 chance:", round(1 / 9, 4), "| pairwise chance: 0.5 | closed R@1 chance:", round(1 / 11, 4))

## 8. Query-level bootstrap %95 güven aralıkları

In [ ]:
CI_METRICS = ["hard_only_recall@1", "pairwise_hard_accuracy", "contrast_consistency", "recall@1", "mrr@10", "ndcg@10"]
ci_rows = []
for model_name, result in RESULTS.items():
    for metric in CI_METRICS:
        values = [row[metric] for row in result["per_query"] if row.get(metric) is not None]
        low, high = bootstrap_ci(values, n_boot=N_BOOT, seed=SEED)
        ci_rows.append({"model": model_name, "metric": metric, "mean": np.mean(values), "ci_low": low, "ci_high": high, "n": len(values)})
CI_DF = pd.DataFrame(ci_rows)
display(CI_DF.round(4))
print("UYARI: n=20 olduğu için aralıkların geniş olması beklenir; final 500 sealed test esas rapordur.")

In [ ]:
plot_df = CI_DF[CI_DF.metric.isin(["hard_only_recall@1", "pairwise_hard_accuracy", "recall@1"])].copy()
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, metric in zip(axes, plot_df.metric.unique()):
    part = plot_df[plot_df.metric == metric].sort_values("mean")
    ax.errorbar(part["mean"], part["model"], xerr=[part["mean"] - part["ci_low"], part["ci_high"] - part["mean"]], fmt="o", capsize=3)
    ax.set_title(metric); ax.set_xlim(-0.03, 1.03); ax.grid(axis="x", alpha=.25)
plt.tight_layout(); plt.show()

## 9. Hard-negative subtype analizi

In [ ]:
subtype_rows = []
for model_name, result in RESULTS.items():
    for item in EVAL_ITEMS:
        scores = result["scores"][item["family_id"]]
        gold_score = scores[item["gold_id"]]
        for candidate in item["candidates"]:
            if candidate["role"] != "hard_negative": continue
            margin = gold_score - scores[candidate["id"]]
            subtype_rows.append({
                "model": model_name, "family_id": item["family_id"], "subtype": candidate["subtype"],
                "target_feature": item["target_feature"], "margin": margin,
                "gold_wins": 1.0 if margin > 0 else 0.5 if margin == 0 else 0.0,
            })
SUBTYPE_PAIRS_DF = pd.DataFrame(subtype_rows)
SUBTYPE_DF = (SUBTYPE_PAIRS_DF.groupby(["model", "subtype"])
              .agg(n=("gold_wins", "size"), accuracy=("gold_wins", "mean"), mean_margin=("margin", "mean"))
              .reset_index())
display(SUBTYPE_DF.sort_values(["model", "accuracy"]).round(4))
print("n küçük subtype satırlarını yalnız tanısal okuyun; inferential sonuç değildir.")

In [ ]:
heat = SUBTYPE_DF.pivot(index="subtype", columns="model", values="accuracy")
plt.figure(figsize=(max(7, 1.6 * len(RESULTS)), max(5, .45 * len(heat))))
sns.heatmap(heat, annot=True, fmt=".2f", cmap="RdYlGn", vmin=0, vmax=1)
plt.title("Gold'un hard negatifi geçme oranı"); plt.tight_layout(); plt.show()

## 10. Slice sonuçları

In [ ]:
slice_rows = []
for model_name, result in RESULTS.items():
    for metric in ["hard_only_recall@1", "pairwise_hard_accuracy", "recall@1"]:
        nested = slice_summary(result["per_query"], EVAL_ITEMS, metric=metric)
        for field, groups in nested.items():
            for value, stats in groups.items():
                slice_rows.append({"model": model_name, "metric": metric, "field": field, "value": value, **stats})
SLICE_DF = pd.DataFrame(slice_rows)
display(SLICE_DF.sort_values(["metric", "field", "model", "value"]).round(4))
print("n<5 dilimler yalnız hata bulma amaçlıdır; ayrı paper iddiası kurmayın.")

## 11. Hata analizi: model neyi birinci getirdi?

In [ ]:
item_by_id = {item["family_id"]: item for item in EVAL_ITEMS}
error_rows = []
for model_name, result in RESULTS.items():
    per_query = {row["query_id"]: row for row in result["per_query"]}
    for query_id, ranking in result["run"].items():
        item = item_by_id[query_id]
        candidates = {candidate["id"]: candidate for candidate in item["candidates"]}
        top = candidates[ranking[0]]
        row = per_query[query_id]
        error_rows.append({
            "model": model_name, "family_id": query_id, "correct@1": int(ranking[0] == item["gold_id"]),
            "gold_rank": row["rank"], "hard_rank": row["hard_rank"],
            "hardest_hard_margin": row["hardest_hard_margin"], "target_feature": item["target_feature"],
            "layer": item["layer"], "predicted_role": top["role"], "predicted_subtype": top["subtype"],
            "query": item["query"], "predicted_text": top["text"],
            "gold_text": candidates[item["gold_id"]]["text"],
        })
ERRORS_DF = pd.DataFrame(error_rows)
display(ERRORS_DF[ERRORS_DF["correct@1"] == 0].sort_values(["model", "gold_rank"]).reset_index(drop=True))

## 12. Paired model karşılaştırmaları

In [ ]:
def aligned_values(result, metric):
    return {row["query_id"]: float(row[metric]) for row in result["per_query"]}

comparison_rows, raw_p = [], {}
names = list(RESULTS)
for i, left_name in enumerate(names):
    for right_name in names[i + 1:]:
        for metric in ["hard_only_recall@1", "pairwise_hard_accuracy", "recall@1", "ndcg@10"]:
            left_map, right_map = aligned_values(RESULTS[left_name], metric), aligned_values(RESULTS[right_name], metric)
            ids = sorted(set(left_map) & set(right_map))
            left, right = [left_map[x] for x in ids], [right_map[x] for x in ids]
            boot = paired_bootstrap(left, right, n_boot=N_BOOT, seed=SEED)
            p = approximate_randomization(left, right, n_iter=N_BOOT, seed=SEED)
            key = f"{left_name}__{right_name}__{metric}"
            raw_p[key] = p
            row = {"comparison": f"{left_name} - {right_name}", "metric": metric, "p_randomization": p, **boot}
            if metric in {"hard_only_recall@1", "recall@1"}:
                row.update({f"mcnemar_{k}": v for k, v in mcnemar(left, right).items()})
            comparison_rows.append(row)
adjusted = holm_adjust(raw_p) if raw_p else {}
for row in comparison_rows:
    left, right = row["comparison"].split(" - ")
    row["p_holm"] = adjusted[f"{left}__{right}__{row['metric']}"]
COMPARISONS_DF = pd.DataFrame(comparison_rows)
display(COMPARISONS_DF.round(4))
print("n=20 pilotta p-değerinden çok effect size ve CI yönünü inceleyin.")

## 13. Kritik sözcük / yaklaşık kök ablation

In [ ]:
FOCUS = SUMMARY_DF["hard_only_recall@1"].idxmax()
focus_spec = next(spec for spec in MODEL_SPECS if spec["name"] == FOCUS)
focus_model = load_model(focus_spec)
ablation_sets = {
    "original": EVAL_ITEMS,
    "critical_deleted": ablate_items(EVAL_ITEMS, "critical_deleted"),
    "f5_roots": ablate_items(EVAL_ITEMS, "f5_roots"),
}
ABLATION_RESULTS = {}
for ablation_name, items in ablation_sets.items():
    ABLATION_RESULTS[ablation_name] = score_encoder(
        focus_model, items, focus_spec.get("query_prefix", ""), focus_spec.get("document_prefix", ""),
        batch_size=BATCH_SIZE,
    )
del focus_model; gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
ABLATION_DF = pd.DataFrame({name: result["summary"] for name, result in ABLATION_RESULTS.items()}).T
ablation_metrics = ["hard_only_recall@1", "pairwise_hard_accuracy", "contrast_consistency", "recall@1", "mrr@10", "ndcg@10"]
display(ABLATION_DF[ablation_metrics].round(4))
print("focus model:", FOCUS)

## 14. Full-corpus retrieval + pooling

Bu katman **her zaman** ortak corpus sıralamasını üretir:

- Preview için 20 query, bütün 220 aday içinde aranır.
- Final için 500 sealed query, bütün 5.500 aday içinde aranır.
- `known-gold diagnostic`, yalnız kendi family gold'unun sırasını izler. Yabancı family belgelerinin
  relevance'ı bilinmediği için bu tablo paper'ın resmi full-corpus sonucu değildir.
- BM25, character 3-gram, word overlap ve bütün dense encoder'ların top-20 sonuçları birleştirilerek
  kör insan yargısı için pooling dosyası hazırlanır.
- İnsan-yargılı pooled qrels bulunduğunda resmi Recall/MRR/nDCG/MAP tablosu otomatik hesaplanır.

In [ ]:
POOL_DEPTH = 20

# 14A — Known-gold diagnostic: full corpus gerçekten sıralanır, ama eksik qrels paper metriği diye sunulmaz.
full_diagnostic = {}
full_per_query_frames = []
for model_name, result in RESULTS.items():
    if "full_run" not in result:
        raise RuntimeError(f"{model_name} cache'i full_run içermiyor. Cache'i silip modeli yeniden çalıştırın.")
    summary, rows = evaluate_run(closed_qrels(EVAL_ITEMS), result["full_run"])
    full_diagnostic[model_name] = summary
    frame = pd.DataFrame(rows)
    frame.insert(0, "model", model_name)
    full_per_query_frames.append(frame)

FULL_DIAGNOSTIC_DF = pd.DataFrame(full_diagnostic).T
FULL_DIAGNOSTIC_PER_QUERY_DF = pd.concat(full_per_query_frames, ignore_index=True)
display(FULL_DIAGNOSTIC_DF[[
    "recall@1", "recall@5", "recall@10", "recall@100", "mrr@10", "ndcg@10", "map@10", "mean_rank"
]].sort_values("recall@10", ascending=False).round(4))
print("UYARI: Bu known-gold diagnostic tablosudur; yabancı belgeler yargılanmadan paper metriği değildir.")

# 14B — Sparse + dense top-k pooling havuzu.
SPARSE_FULL_RUNS = full_corpus_sparse_runs(EVAL_ITEMS)
pool_system_runs = {**SPARSE_FULL_RUNS, **{name: result["full_run"] for name, result in RESULTS.items()}}
pool_rows = build_pool_rows(EVAL_ITEMS, pool_system_runs, depth=POOL_DEPTH)

POOL_DF = pd.DataFrame(pool_rows)
POOL_FILE = OUTPUT_DIR / "pooling_judgment_template.jsonl"
with POOL_FILE.open("w", encoding="utf-8") as handle:
    for row in pool_rows:
        handle.write(json.dumps(row, ensure_ascii=False) + "\n")
display(POOL_DF.groupby("query_id").size().describe().to_frame("pooled_docs_per_query"))
print("kör pooling şablonu:", POOL_FILE)

# 14C — İnsan relevance yargıları varsa resmi full-corpus metrikleri.
FULL_PAPER_DF = pd.DataFrame()
FULL_PAPER_PER_QUERY_DF = pd.DataFrame()
if not POOLED_QRELS.exists():
    print("SKIP official full-corpus metrics: insan-yargılı pooled qrels henüz yok.")
else:
    pooled_qrels = load_qrels(POOLED_QRELS)
    judged_rows = sum(len(rels) for rels in pooled_qrels.values())
    has_nonrelevant = any(score <= 0 for rels in pooled_qrels.values() for score in rels.values())
    if judged_rows <= len(EVAL_ITEMS) or not has_nonrelevant:
        raise ValueError("Qrels own-gold-only görünüyor; resmi full-corpus metriği hesaplanmadı.")
    paper_summaries, paper_frames = {}, []
    for model_name, result in RESULTS.items():
        summary, rows = evaluate_run(pooled_qrels, result["full_run"])
        paper_summaries[model_name] = summary
        frame = pd.DataFrame(rows); frame.insert(0, "model", model_name); paper_frames.append(frame)
    FULL_PAPER_DF = pd.DataFrame(paper_summaries).T
    FULL_PAPER_PER_QUERY_DF = pd.concat(paper_frames, ignore_index=True)
    display(FULL_PAPER_DF[[
        "recall@1", "recall@5", "recall@10", "recall@100", "mrr@10", "ndcg@10", "map@10", "mean_rank"
    ]].round(4))

## 15. Sonuçları dışa aktar

In [ ]:
SUMMARY_DF.to_csv(OUTPUT_DIR / "encoder_summary.csv", index_label="model")
FULL_DIAGNOSTIC_DF.to_csv(OUTPUT_DIR / "full_corpus_known_gold_diagnostic.csv", index_label="model")
FULL_DIAGNOSTIC_PER_QUERY_DF.to_csv(OUTPUT_DIR / "full_corpus_known_gold_per_query.csv", index=False)
if not FULL_PAPER_DF.empty:
    FULL_PAPER_DF.to_csv(OUTPUT_DIR / "full_corpus_pooled_human_summary.csv", index_label="model")
    FULL_PAPER_PER_QUERY_DF.to_csv(OUTPUT_DIR / "full_corpus_pooled_human_per_query.csv", index=False)
ARTIFACT_DF.to_csv(OUTPUT_DIR / "artifact_baselines.csv", index_label="baseline")
CI_DF.to_csv(OUTPUT_DIR / "bootstrap_ci.csv", index=False)
SUBTYPE_DF.to_csv(OUTPUT_DIR / "hard_subtype_results.csv", index=False)
SLICE_DF.to_csv(OUTPUT_DIR / "slice_results.csv", index=False)
ERRORS_DF.to_csv(OUTPUT_DIR / "error_analysis.csv", index=False)
COMPARISONS_DF.to_csv(OUTPUT_DIR / "paired_comparisons.csv", index=False)
ABLATION_DF.to_csv(OUTPUT_DIR / "ablations.csv", index_label="ablation")
pd.DataFrame(RUNTIME_ROWS).to_csv(OUTPUT_DIR / "runtime.csv", index=False)
(OUTPUT_DIR / "model_errors.json").write_text(json.dumps(MODEL_ERRORS, ensure_ascii=False, indent=2), encoding="utf-8")
metadata = {
    "dataset_mode": DATASET_MODE, "dataset_file": str(DATA_FILE), "dataset_sha256": DATA_SHA256,
    "evaluation_code_sha256": EVAL_CODE_SHA256, "evaluation_api": EVALUATION_API_VERSION,
    "family_count": len(EVAL_ITEMS), "seed": SEED, "bootstrap_draws": N_BOOT,
    "model_specs": MODEL_SPECS, "notice": DATA_NOTICE,
}
(OUTPUT_DIR / "run_metadata.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
archive = shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=OUTPUT_DIR)
print("çıktı:", OUTPUT_DIR)
print("zip:", archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    pass

## 16. Pilot sonucu yorumlama sırası

1. Önce ucuz baseline'ların yüksek olup olmadığına bakın; yüksekse veri artefaktı vardır.
2. Encoder'larda önce `hard_only_recall@1`, `pairwise_hard_accuracy` ve margin'leri okuyun.
3. Hangi hard subtype'ların sürekli kaybedildiğini inceleyin.
4. Kritik sözcük silinince skorun düşmesi morfolojik sinyale duyarlılıkla uyumludur; tek başına nedensellik kanıtı değildir.
5. Bu 20 örnekle model seçimini kesinleştirmeyin. Ayarları 100 development'ta seçin; 500 sealed
   testte CI + paired testlerle final sonucu raporlayın.
6. `full_corpus_known_gold_diagnostic.csv` yalnız tanısaldır. Paper'da resmi full-corpus sonucu için pooling şablonunu insanlar yargılamalı ve pooled qrels ile yeniden çalıştırmalısınız.
7. Fine-tuning seed varyansı frozen encoder baseline'ına uygulanmaz; training aşamasında en az üç seed ayrıca raporlanmalı.